# 01. Titanic ベースライン

流れ: 環境準備 → データ取得 → EDA → 前処理 → ベースラインモデル → 提出ファイル作成

事前準備:
- Kaggle の Titanic コンペでルールに同意（Join）しておく
- Colab の「シークレット」に `KAGGLE_USERNAME` と `KAGGLE_KEY` を登録し、ノートブックからのアクセスをオンにする

## 1. 環境準備（Drive マウントと Kaggle 認証）

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

# Kaggle 認証（キーはコードに直書きしない）
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

BASE = '/content/drive/MyDrive/kaggle-exercise'
DATA_DIR = f'{BASE}/data/titanic'
OUT_DIR = f'{BASE}/outputs/titanic'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

## 2. データのダウンロード（初回のみ）

In [ ]:
if not os.path.exists(f'{DATA_DIR}/train.csv'):
    !kaggle competitions download -c titanic -p {DATA_DIR}
    !unzip -o {DATA_DIR}/titanic.zip -d {DATA_DIR}
else:
    print('Drive に既にデータがあります。ダウンロードをスキップします。')

!ls -la {DATA_DIR}

## 3. データの読み込みと基本確認

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(train.shape, test.shape)
train.head()

In [ ]:
train.info()
print()
print('欠損数:')
print(train.isnull().sum())
train.describe()

## 4. EDA（探索的データ分析）

考えてみる問い: どの属性で生存率が大きく違うか？ 欠損の多い列はどう扱うか？

In [ ]:
print('全体の生存率:', round(train['Survived'].mean(), 3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['Sex', 'Pclass', 'Embarked']):
    train.groupby(col)['Survived'].mean().plot(kind='bar', ax=ax)
    ax.set_title(f'{col} ごとの生存率')
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train.loc[train['Survived'] == 1, 'Age'].plot(kind='hist', bins=20, alpha=0.6, ax=axes[0], label='生存')
train.loc[train['Survived'] == 0, 'Age'].plot(kind='hist', bins=20, alpha=0.6, ax=axes[0], label='死亡')
axes[0].set_title('年齢分布')
axes[0].legend()

train.boxplot(column='Fare', by='Survived', ax=axes[1])
axes[1].set_title('運賃と生存')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 5. 前処理と特徴量

まずは最小限のベースラインを作る。特徴量の追加は次の演習で行う。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Pclass', 'Sex', 'Embarked']

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), cat_cols),
])

X = train[num_cols + cat_cols]
y = train['Survived']
X_test = test[num_cols + cat_cols]

## 6. ベースラインモデルと交差検証

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    print(f'{name}: 平均 {scores.mean():.4f} (標準偏差 {scores.std():.4f})')

## 7. 提出ファイルの作成（Drive に保存）

In [ ]:
best = Pipeline([('prep', preprocess), ('model', models['RandomForest'])])
best.fit(X, y)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': best.predict(X_test),
})
submission.to_csv(f'{OUT_DIR}/submission_baseline.csv', index=False)
submission.head()

## 次のステップ

- 提出ファイルを Kaggle にアップロードしてスコアを確認する
- `Name` から敬称（Mr, Mrs, Miss など）を抽出する
- `SibSp + Parch + 1` で家族人数を作る
- `Cabin` の頭文字や欠損フラグを使う
- 特徴量を変えるたびに交差検証スコアを記録する